# Wind取数尝试

Update: 20260529

## Wind MySQL 数据库连接复核

下面的代码用于复核 Wind MySQL 数据库连接过程。账号、密码、主机、端口和数据库名从环境变量读取（见仓库根目录 `.env.example`），不再写入代码。

In [ ]:
import pandas as pd
import os
import pymysql

MYSQL_HOST = os.environ.get("WIND_HOST")
MYSQL_PORT = int(os.environ.get("WIND_PORT", "3306"))
MYSQL_USER = os.environ.get("WIND_USER")
MYSQL_PASSWORD = os.environ.get("WIND_PASSWORD")
MYSQL_DATABASE = os.environ.get("WIND_DATABASE")

conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
    charset="utf8mb4",
    connect_timeout=10,
    read_timeout=100,
    write_timeout=10,
    cursorclass=pymysql.cursors.DictCursor,
)

def query_df(sql, params=None):
    with conn.cursor() as cursor:
        cursor.execute(sql, params or {})
        rows = cursor.fetchall()
    return pd.DataFrame(rows)

In [ ]:
# 轻量连接测试：返回 MySQL 版本、当前用户和当前数据库
connection_check = query_df(
    """
    select
        version() as mysql_version,
        current_user() as current_user_name,
        database() as current_database
    """,
)

connection_check

## 查询 SQL 复核

目标：查询贵州茅台 `600519.SH` 在 `2026-05-28` 的 A 股日行情收盘价。

使用表：`ashareeodprices`  
关键字段：`s_info_windcode`、`trade_dt`、`s_dq_close`

复核 SQL：

```sql
select s_info_windcode, trade_dt, s_dq_close
from ashareeodprices
where s_info_windcode = '600519.SH'
  and trade_dt = '20260528'
limit 1;
```

我此前实测该 SQL 返回 1 条记录，`s_dq_close = 1275.9800`。

In [ ]:
count_sql = """
select count(*) as n
from ashareeodprices
where s_info_windcode = %(windcode)s
  and trade_dt = %(trade_dt)s
"""

price_sql = """
select s_info_windcode, trade_dt, s_dq_close
from ashareeodprices
where s_info_windcode = %(windcode)s
  and trade_dt = %(trade_dt)s
limit 1
"""

In [ ]:
params = {"windcode": "600519.SH", "trade_dt": "20260706"}

row_count = query_df(count_sql, params=params)
close_price = query_df(price_sql, params=params)

display(row_count)
display(close_price)

### 更大规模的数据测试

提高read_timeout的参数设置即可

In [ ]:
price_sql = """
select
    f_info_windcode,
    price_date,
    ann_date,
    f_nav_unit,
    f_nav_accumulated,
    f_nav_adjfactor,
    f_nav_adjusted,
    round(f_nav_unit * f_nav_adjfactor, 8) as calculated_adjusted_nav,
    round(
        f_nav_adjusted - f_nav_unit * f_nav_adjfactor,
        8
    ) as adjusted_nav_difference
from chinamutualfundnav
where f_info_windcode = '001722.OF'
  and price_date between '20260724' and '20260728'
order by price_date;
"""

In [ ]:
params = {"windcode": "600519.SH"}
close_price = query_df(price_sql)

In [ ]:
close_price

### 测试基金数据添加索引效果

In [ ]:
params = {"windcode": "000001.OF", "price_date": "20260706"}

In [ ]:
# price_sql = """
# select f_info_windcode, price_date, f_nav_unit
# from chinamutualfundnav
# where f_info_windcode = %(windcode)s 
#     and price_date = %(price_date)s
# """

In [ ]:
price_sql = """
select f_info_windcode, price_date, f_nav_unit
from chinamutualfundnav
where  price_date >= '20250101' and price_date <= '20251231'
"""

In [ ]:
    and 


In [ ]:
df = query_df(price_sql, params=params)

In [ ]:
df